In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.11.0+cu128
True
12.8
NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
import torch

print("GPU Count:", torch.cuda.device_count())
print("Current GPU:", torch.cuda.current_device())
print("GPU Name:", torch.cuda.get_device_name(0))

GPU Count: 1
Current GPU: 0
GPU Name: NVIDIA GeForce RTX 4050 Laptop GPU


In [4]:
import os
import time
import json
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [5]:
# Imports
BASE_DIR = r"E:\SmartVision-AI\data"
CLASSIFICATION_DIR = os.path.join(BASE_DIR, "classification")

TRAIN_DIR = os.path.join(CLASSIFICATION_DIR, "train")
VAL_DIR = os.path.join(CLASSIFICATION_DIR, "val")
TEST_DIR = os.path.join(CLASSIFICATION_DIR, "test")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Using device: cuda
NVIDIA GeForce RTX 4050 Laptop GPU


In [6]:
# Paths + Device

train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [7]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

BASE_DIR = r"E:\SmartVision-AI\data"
CLASSIFICATION_DIR = os.path.join(BASE_DIR, "classification")

TRAIN_DIR = os.path.join(CLASSIFICATION_DIR, "train")
VAL_DIR = os.path.join(CLASSIFICATION_DIR, "val")
TEST_DIR = os.path.join(CLASSIFICATION_DIR, "test")

train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_test_transforms)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", num_classes)
print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))
print(class_names)

Classes: 26
Train: 1820
Val: 390
Test: 390
['airplane', 'bed', 'bench', 'bicycle', 'bird', 'bottle', 'bowl', 'bus', 'cake', 'car', 'cat', 'chair', 'couch', 'cow', 'cup', 'dog', 'elephant', 'horse', 'motorcycle', 'person', 'pizza', 'potted plant', 'stop sign', 'traffic light', 'train', 'truck']


In [8]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# Freeze feature extractor
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier for 26 classes
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

print(model.classifier)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to C:\Users\athiy/.cache\torch\hub\checkpoints\mobilenet_v2-7ebf99e0.pth


100%|█████████████████████████████████████████████████████████████████████████████| 13.6M/13.6M [00:03<00:00, 3.86MB/s]


Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=26, bias=True)
)


In [9]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10):
    best_val_acc = 0.0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        print("-" * 30)

        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)

            train_loss += loss.item() * images.size(0)
            train_correct += torch.sum(preds == labels).item()
            train_total += labels.size(0)

        train_epoch_loss = train_loss / train_total
        train_epoch_acc = train_correct / train_total

        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                _, preds = torch.max(outputs, 1)

                val_loss += loss.item() * images.size(0)
                val_correct += torch.sum(preds == labels).item()
                val_total += labels.size(0)

        val_epoch_loss = val_loss / val_total
        val_epoch_acc = val_correct / val_total

        history["train_loss"].append(train_epoch_loss)
        history["train_acc"].append(train_epoch_acc)
        history["val_loss"].append(val_epoch_loss)
        history["val_acc"].append(val_epoch_acc)

        print(f"Train Loss: {train_epoch_loss:.4f} | Train Acc: {train_epoch_acc:.4f}")
        print(f"Val Loss:   {val_epoch_loss:.4f} | Val Acc:   {val_epoch_acc:.4f}")

        if val_epoch_acc > best_val_acc:
            best_val_acc = val_epoch_acc
            torch.save(model.state_dict(), r"E:\SmartVision-AI\models\mobilenetv2_best.pth")
            print("✅ Best model saved")

    return history

In [10]:
start_time = time.time()

history = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs=10
)

end_time = time.time()
training_time = end_time - start_time

print(f"Training Time: {training_time/60:.2f} minutes")


Epoch 1/10
------------------------------
Train Loss: 2.8785 | Train Acc: 0.2775
Val Loss:   2.4722 | Val Acc:   0.4846
✅ Best model saved

Epoch 2/10
------------------------------
Train Loss: 2.1745 | Train Acc: 0.5022
Val Loss:   2.0289 | Val Acc:   0.5462
✅ Best model saved

Epoch 3/10
------------------------------
Train Loss: 1.8372 | Train Acc: 0.5681
Val Loss:   1.8130 | Val Acc:   0.5718
✅ Best model saved

Epoch 4/10
------------------------------
Train Loss: 1.6326 | Train Acc: 0.6082
Val Loss:   1.6640 | Val Acc:   0.5744
✅ Best model saved

Epoch 5/10
------------------------------
Train Loss: 1.5008 | Train Acc: 0.6187
Val Loss:   1.5683 | Val Acc:   0.6000
✅ Best model saved

Epoch 6/10
------------------------------
Train Loss: 1.4097 | Train Acc: 0.6330
Val Loss:   1.4973 | Val Acc:   0.6051
✅ Best model saved

Epoch 7/10
------------------------------
Train Loss: 1.3175 | Train Acc: 0.6659
Val Loss:   1.4542 | Val Acc:   0.5872

Epoch 8/10
---------------------------

In [11]:
for param in model.features[-4:].parameters():
    param.requires_grad = True

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)

history_finetune = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs=5
)


Epoch 1/5
------------------------------
Train Loss: 0.9928 | Train Acc: 0.7396
Val Loss:   1.2053 | Val Acc:   0.6487
✅ Best model saved

Epoch 2/5
------------------------------
Train Loss: 0.8914 | Train Acc: 0.7582
Val Loss:   1.1291 | Val Acc:   0.6795
✅ Best model saved

Epoch 3/5
------------------------------
Train Loss: 0.7955 | Train Acc: 0.7764
Val Loss:   1.0841 | Val Acc:   0.7000
✅ Best model saved

Epoch 4/5
------------------------------
Train Loss: 0.7125 | Train Acc: 0.8027
Val Loss:   1.0613 | Val Acc:   0.7051
✅ Best model saved

Epoch 5/5
------------------------------
Train Loss: 0.6436 | Train Acc: 0.8225
Val Loss:   1.0485 | Val Acc:   0.7026


In [12]:
model.load_state_dict(torch.load(r"E:\SmartVision-AI\models\mobilenetv2_best.pth"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = accuracy_score(all_labels, all_preds)

print("MobileNetV2 Test Accuracy:", round(test_acc, 4))
print()
print(classification_report(all_labels, all_preds, target_names=class_names))

MobileNetV2 Test Accuracy: 0.7205

               precision    recall  f1-score   support

     airplane       0.87      0.87      0.87        15
          bed       0.80      0.53      0.64        15
        bench       0.53      0.60      0.56        15
      bicycle       0.80      0.80      0.80        15
         bird       0.82      0.93      0.88        15
       bottle       0.69      0.73      0.71        15
         bowl       0.57      0.53      0.55        15
          bus       0.75      0.80      0.77        15
         cake       0.83      0.67      0.74        15
          car       0.65      1.00      0.79        15
          cat       0.65      0.73      0.69        15
        chair       0.67      0.67      0.67        15
        couch       0.69      0.60      0.64        15
          cow       0.94      1.00      0.97        15
          cup       0.52      0.80      0.63        15
          dog       0.73      0.53      0.62        15
     elephant       1.00     

In [13]:
mobilenet_results = {
    "model": "MobileNetV2",
    "test_accuracy": test_acc,
    "macro_f1": 0.72,
    "precision_macro": 0.74,
    "recall_macro": 0.72,
    "training_time_minutes": round(training_time / 60, 2),
    "num_classes": num_classes
}

with open(r"E:\SmartVision-AI\models\mobilenetv2_metrics.json", "w") as f:
    json.dump(mobilenet_results, f, indent=4)

print("MobileNetV2 metrics saved")

MobileNetV2 metrics saved
